# Stock Market Data Pipeline
## Project: How Oil Price Movements Affect Stock Sectors

This notebook pulls, processes, and validates stock market data from Yahoo Finance.  

### What this notebook does:
1. **Fetches** historical price data for 30 tickers across 6 sectors (3 years, daily)
2. **Calculates** derived financial metrics per ticker
3. **Fetches** company metadata (PE ratio, market cap, beta, margins, etc.)
4. **Builds** an oil price correlation table — the core of the analysis
5. **Aggregates** daily data at the sector level
6. **Validates** data quality per ticker and flags issues
7. **Exports** 5  CSV files 

### Output Files:
| File | Description |
|---|---|
| `stock_prices.csv` | Main fact table — daily OHLCV + derived metrics |
| `company_info.csv` | Dimension table — 1 row per company |
| `oil_correlations.csv` | Correlation of every stock vs. oil price |
| `sector_summary.csv` | Daily aggregated returns per sector |
| `data_validation.csv` | Data quality report per ticker |


---
## Section 1 — Imports & Configuration

We use:
- **`yfinance`** — free Yahoo Finance API wrapper, no API key needed
- **`pandas`** — data manipulation
- **`numpy`** — numerical calculations
- **`warnings`** — to suppress non-critical yfinance messages


In [1]:
!pip install yfinance 

import yfinance as yf
import pandas as pd
import numpy as np
import itables
from datetime import datetime
import os
import warnings
warnings.filterwarnings("ignore")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Section 2 — Ticker Configuration

Tickers are organized intentionally by sector — not randomly chosen.  
Each sector serves a specific analytical purpose in the oil impact study:

| Sector | Purpose |
|---|---|
| Oil & Energy | The independent variable — oil price is the core of our analysis |
| Technology | Expected low/negative correlation with oil |
| Finance | Sensitive to interest rates and economic stability |
| Healthcare | Defensive sector — typically stable during oil shocks |
| Consumer | Mixed sensitivity depending on oil's effect on spending |
| Gulf Market | Direct regional exposure to oil price movements |
| Benchmarks | Reference points (S&P 500, VIX, Gold) for context |

**Key parameters:**
- `PERIOD = "3y"` → ~750 trading days per ticker → ~22,500 total rows
- `INTERVAL = "1d"` → daily granularity (best balance of detail vs. noise)


In [2]:
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PERIOD   = "3y"   # 3 years of daily data
INTERVAL = "1d"   # daily bars

SECTORS = {

    "Oil & Energy": {
        "CL=F":  "Crude Oil Futures (WTI)",      # THE benchmark — core of the analysis
        "BZ=F":  "Brent Crude Futures",           # international oil benchmark
        "XOM":   "ExxonMobil",                    # largest US oil major
        "CVX":   "Chevron",                       # second largest US oil major
        "SLB":   "SLB (Schlumberger)",            # oilfield services leader
        "XLE":   "Energy Sector ETF",             # tracks whole energy sector
    },

    "Technology": {
        "AAPL":  "Apple",
        "MSFT":  "Microsoft",
        "NVDA":  "NVIDIA",
        "GOOGL": "Alphabet (Google)",
        "XLK":   "Tech Sector ETF",
    },

    "Finance": {
        "JPM":   "JPMorgan Chase",
        "BAC":   "Bank of America",
        "GS":    "Goldman Sachs",
        "XLF":   "Finance Sector ETF",
    },

    "Healthcare": {
        "JNJ":   "Johnson & Johnson",
        "PFE":   "Pfizer",
        "UNH":   "UnitedHealth Group",
        "XLV":   "Healthcare Sector ETF",
    },

    "Consumer": {
        "AMZN":  "Amazon",
        "WMT":   "Walmart",
        "MCD":   "McDonald's",
        "XLY":   "Consumer Discretionary ETF",
    },

    "Gulf Market": {
        "2222.SR": "Saudi Aramco (Tadawul)",
        "2010.SR": "SABIC (Saudi Petrochemicals)",
        "1180.SR": "Al Rajhi Bank (Saudi)",
    },

    "Benchmarks": {
        "^GSPC":  "S&P 500 Index",
        "^IXIC":  "NASDAQ Composite",
        "GC=F":   "Gold Futures",
        "^VIX":   "VIX Volatility Index",
    },
}

# Flatten: { ticker: name }
ALL_TICKERS = {
    ticker: name
    for sector_dict in SECTORS.values()
    for ticker, name in sector_dict.items()
}

# Reverse map: { ticker: sector }
TICKER_TO_SECTOR = {
    ticker: sector
    for sector, sector_dict in SECTORS.items()
    for ticker in sector_dict
}

print(f"✅ Configuration ready")
print(f"   Total tickers : {len(ALL_TICKERS)}")
print(f"   Sectors       : {len(SECTORS)}")
print(f"   Period        : {PERIOD}  |  Interval: {INTERVAL}")
print(f"   Output folder : ./{OUTPUT_DIR}/")


✅ Configuration ready
   Total tickers : 30
   Sectors       : 7
   Period        : 3y  |  Interval: 1d
   Output folder : ./data/


---
## Section 3 — Fetching Price History (OHLCV + Derived Metrics)

For each ticker, we pull daily OHLCV data and calculate the following derived columns:

| Column | Formula | What it tells you |
|---|---|---|
| `Daily_Return_%` | `(Close - Prev_Close) / Prev_Close × 100` | Day-over-day price change |
| `Intraday_Range` | `High - Low` | Intraday volatility in price units |
| `Intraday_Range_%` | `Intraday_Range / Close × 100` | Intraday volatility as % of price |
| `Volatility_30D` | Rolling 30-day std dev of daily returns | Risk measure — higher = more volatile |
| `MA_30D` / `MA_90D` | Rolling 30/90-day average of Close | Trend smoothing |
| `Cumulative_Return_%` | `(Close - First_Close) / First_Close × 100` | Total return from start of period |
| `Pct_Above_MA30` | `(Close - MA_30D) / MA_30D × 100` | Is stock above or below its trend? |
| `Volume_Spike` | `1 if Volume > 2× Avg_Volume_30D` | Flag for unusual trading activity |

> **Note on `auto_adjust=True`:** Prices are automatically adjusted for stock splits  
> and dividend distributions, ensuring historical comparisons are accurate.


In [3]:
def fetch_price_history():
    print(f"Fetching price history for {len(ALL_TICKERS)} tickers | Period: {PERIOD}\n")

    all_frames = []
    success, failed = [], []

    for ticker, name in ALL_TICKERS.items():
        try:
            df = yf.download(
                ticker,
                period=PERIOD,
                interval=INTERVAL,
                auto_adjust=True,   # adjusts for splits & dividends
                progress=False
            )

            if df.empty:
                print(f"  ⚠️  NO DATA  — {ticker}")
                failed.append(ticker)
                continue

            # Flatten multi-level column index (yfinance sometimes returns tuples)
            df = df.reset_index()
            df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

            # Identity columns
            df["Ticker"] = ticker
            df["Name"]   = name
            df["Sector"] = TICKER_TO_SECTOR[ticker]

            # Sort chronologically before any rolling calculations
            df = df.sort_values("Date").reset_index(drop=True)

            # ── Derived metrics ────────────────────────────────────────────
            df["Daily_Return_%"]      = df["Close"].pct_change() * 100
            df["Intraday_Range"]      = df["High"] - df["Low"]
            df["Intraday_Range_%"]    = (df["Intraday_Range"] / df["Close"]) * 100
            df["Volatility_30D"]      = df["Daily_Return_%"].rolling(30).std()
            df["Avg_Volume_30D"]      = df["Volume"].rolling(30).mean()
            df["MA_30D"]              = df["Close"].rolling(30).mean()
            df["MA_90D"]              = df["Close"].rolling(90).mean()
            df["Cumulative_Return_%"] = ((df["Close"] - df["Close"].iloc[0]) / df["Close"].iloc[0]) * 100
            df["Pct_Above_MA30"]      = ((df["Close"] - df["MA_30D"]) / df["MA_30D"]) * 100
            df["Prev_Close"]          = df["Close"].shift(1)
            df["Price_Change"]        = df["Close"] - df["Prev_Close"]
            df["Volume_Spike"]        = (df["Volume"] > df["Avg_Volume_30D"] * 2).astype(int)

            all_frames.append(df)
            print(f"  ✅  {ticker:<12} {name:<35} {len(df):>4} rows")
            success.append(ticker)

        except Exception as e:
            print(f"  ❌  FAILED — {ticker}: {e}")
            failed.append(ticker)

    print(f"\nDone — {len(success)} succeeded | {len(failed)} failed")
    if failed:
        print(f"Failed: {', '.join(failed)}")

    combined = pd.concat(all_frames, ignore_index=True)

    # Round all floats to 4 decimal places for cleaner CSV output
    float_cols = combined.select_dtypes(include="float64").columns
    combined[float_cols] = combined[float_cols].round(4)

    return combined, success, failed

# ── Run ───────────────────────────────────────────────────────────
prices_df, success_list, failed_list = fetch_price_history()

print(f"\n📊 Price history shape : {prices_df.shape}")
print(f"   Date range         : {prices_df['Date'].min().date()} → {prices_df['Date'].max().date()}")
print(f"   Columns            : {list(prices_df.columns)}")


Fetching price history for 30 tickers | Period: 3y

  ✅  CL=F         Crude Oil Futures (WTI)              756 rows
  ✅  BZ=F         Brent Crude Futures                  757 rows
  ✅  XOM          ExxonMobil                           753 rows
  ✅  CVX          Chevron                              753 rows
  ✅  SLB          SLB (Schlumberger)                   753 rows
  ✅  XLE          Energy Sector ETF                    753 rows
  ✅  AAPL         Apple                                753 rows
  ✅  MSFT         Microsoft                            753 rows
  ✅  NVDA         NVIDIA                               753 rows
  ✅  GOOGL        Alphabet (Google)                    753 rows
  ✅  XLK          Tech Sector ETF                      753 rows
  ✅  JPM          JPMorgan Chase                       753 rows
  ✅  BAC          Bank of America                      753 rows
  ✅  GS           Goldman Sachs                        753 rows
  ✅  XLF          Finance Sector ETF                

In [4]:
# Quick preview of the price data
itables.show(prices_df)


Loading ITables v2.5.2 from the internet... (need help?)


---
## Section 4 — Fetching Company Metadata

This pulls static/semi-static information about each company from Yahoo Finance.  
Unlike price data (which changes daily), metadata updates less frequently.

**Reliability notes:**
| Field | Reliability | Notes |
|---|---|---|
| `beta`, `dividendYield`, `52WeekHigh/Low` | ✅ High | Calculated from price data |
| `marketCap`, `trailingPE` | ⚠️ Medium | Updated periodically, not real-time |
| `forwardPE`, `targetMeanPrice` | ⚠️ Medium | Analyst estimates — can vary by source |
| `recommendationMean` | ⚠️ Medium | May be missing for Gulf/less-covered stocks |

> **`Analyst_Upside_%`** is a derived column we calculate:  
> `(Target Price - Current Price) / Current Price × 100`  
> It shows how much upside analysts expect for each stock.


In [5]:
def fetch_company_info(successful_tickers):
    print(f"Fetching metadata for {len(successful_tickers)} tickers...\n")

    records = []
    fields = [
        "shortName", "longName", "sector", "industry",
        "country", "currency", "exchange",
        "marketCap", "enterpriseValue",
        "trailingPE", "forwardPE", "priceToBook", "priceToSalesTrailing12Months",
        "dividendYield", "dividendRate", "payoutRatio",
        "beta",
        "fiftyTwoWeekHigh", "fiftyTwoWeekLow",
        "fiftyDayAverage", "twoHundredDayAverage",
        "averageVolume", "averageVolume10days",
        "returnOnEquity", "returnOnAssets",
        "debtToEquity", "currentRatio",
        "revenueGrowth", "earningsGrowth",
        "totalRevenue", "grossMargins", "operatingMargins", "profitMargins",
        "recommendationMean", "numberOfAnalystOpinions",
        "targetMeanPrice", "currentPrice",
    ]

    for ticker in successful_tickers:
        try:
            info = yf.Ticker(ticker).info
            row  = {
                "Ticker": ticker,
                "Name":   ALL_TICKERS.get(ticker, ""),
                "Sector": TICKER_TO_SECTOR.get(ticker, ""),
            }
            for f in fields:
                row[f] = info.get(f, None)

            records.append(row)
            print(f"  ✅  {ticker:<12} {str(row.get('shortName','N/A')):<30} | {row.get('sector','N/A')}")

        except Exception as e:
            print(f"  ❌  {ticker}: {e}")

    df = pd.DataFrame(records)

    # Derived: analyst upside potential
    if "targetMeanPrice" in df.columns and "currentPrice" in df.columns:
        df["Analyst_Upside_%"] = (
            (df["targetMeanPrice"] - df["currentPrice"]) / df["currentPrice"] * 100
        ).round(2)

    return df

# ── Run ───────────────────────────────────────────────────────────
info_df = fetch_company_info(success_list)

print(f"\n📊 Metadata shape : {info_df.shape}")


Fetching metadata for 30 tickers...

  ✅  CL=F         Crude Oil May 26               | None
  ✅  BZ=F         Brent Crude Oil Last Day Financ | None
  ✅  XOM          Exxon Mobil Corporation        | Energy
  ✅  CVX          Chevron Corporation            | Energy
  ✅  SLB          SLB Limited                    | Energy
  ✅  XLE          State Street Energy Select Sect | None
  ✅  AAPL         Apple Inc.                     | Technology
  ✅  MSFT         Microsoft Corporation          | Technology
  ✅  NVDA         NVIDIA Corporation             | Technology
  ✅  GOOGL        Alphabet Inc.                  | Communication Services
  ✅  XLK          State Street Technology Select  | None
  ✅  JPM          JP Morgan Chase & Co.          | Financial Services
  ✅  BAC          Bank of America Corporation    | Financial Services
  ✅  GS           Goldman Sachs Group, Inc. (The) | Financial Services
  ✅  XLF          State Street Financial Select S | None
  ✅  JNJ          Johnson & Johnso

In [6]:
# Preview metadata
itables.show(info_df[["Ticker", "Name", "Sector", "marketCap", "beta",
         "trailingPE", "dividendYield", "profitMargins", "Analyst_Upside_%"]])


Loading ITables v2.5.2 from the internet... (need help?)


---
## Section 5 — Oil Price Correlation Analysis

This is the **analytical core** of the project.

We calculate the **Pearson correlation coefficient** between each stock's daily returns  
and WTI crude oil (`CL=F`) daily returns. The result is a number between -1 and +1:

| Range | Meaning |
|---|---|
| `0.5 → 1.0` | Strong Positive — stock moves with oil |
| `0.25 → 0.5` | Moderate Positive |
| `-0.25 → 0.25` | Weak / No Relationship |
| `-0.5 → -0.25` | Moderate Negative |
| `-1.0 → -0.5` | Strong Negative — stock moves against oil |

We calculate **two correlations** per stock:
- **`Oil_Correlation_Full`** — over the entire 3-year period
- **`Oil_Correlation_30D`** — over the most recent 30 trading days only

The difference between them (`Correlation_Shift`) shows whether the relationship  
between oil and a stock has **changed recently** — a key insight for the project narrative.


In [7]:
def build_oil_correlation(prices_df):
    print("Building oil correlation table...\n")

    # Pivot: rows = Date, columns = Ticker, values = Daily_Return_%
    pivot = prices_df.pivot_table(
        index="Date", columns="Ticker", values="Daily_Return_%"
    )

    if "CL=F" not in pivot.columns:
        print("⚠️  Oil ticker CL=F not found — skipping correlation step")
        return pd.DataFrame()

    oil_returns = pivot["CL=F"].dropna()
    records = []

    for ticker in pivot.columns:
        if ticker == "CL=F":
            continue

        stock_returns = pivot[ticker].dropna()
        common_dates  = oil_returns.index.intersection(stock_returns.index)

        # Minimum data threshold
        if len(common_dates) < 60:
            print(f"  ⚠️  {ticker}: only {len(common_dates)} overlapping days — skipped")
            continue

        oil_s   = oil_returns.loc[common_dates]
        stock_s = stock_returns.loc[common_dates]

        # ── Correlations ───────────────────────────────
        corr_full = stock_s.corr(oil_s)

        recent_df = pd.DataFrame({
            "oil": oil_s,
            "stock": stock_s
        }).dropna().tail(30)

        corr_recent = (
            recent_df["stock"].corr(recent_df["oil"])
            if len(recent_df) >= 15 else None
        )

        # ── Correlation Shift (FIXED) ─────────────────
        corr_shift = (
            round(corr_recent - corr_full, 4)
            if corr_recent is not None else None
        )

        # ── Strength Classification (Reusable) ───────
        def classify_corr(corr):
            if corr is None:
                return None
            if corr >= 0.5:
                return "Strong Positive"
            elif corr >= 0.25:
                return "Moderate Positive"
            elif corr >= -0.25:
                return "Weak/No Relation"
            elif corr >= -0.5:
                return "Moderate Negative"
            else:
                return "Strong Negative"

        # ── Record ───────────────────────────────────
        records.append({
            "Ticker": ticker,
            "Name": ALL_TICKERS.get(ticker, ""),
            "Sector": TICKER_TO_SECTOR.get(ticker, ""),
            "Oil_Correlation_Full": round(corr_full, 4),
            "Oil_Correlation_30D": round(corr_recent, 4) if corr_recent is not None else None,
            "Correlation_Shift": corr_shift,
            "Data_Days": len(common_dates),
            "Corr_Strength": classify_corr(corr_full),
            "Corr_Strength_30D": classify_corr(corr_recent),
        })

        # ── Logging ──────────────────────────────────
        recent_str = f"{corr_recent:>+.4f}" if corr_recent is not None else "N/A"
        print(f"  ✅  {ticker:<12} Full: {corr_full:>+.4f}  |  Recent 30D: {recent_str}")

    return pd.DataFrame(records)

corr_df = build_oil_correlation(prices_df)



Building oil correlation table...

  ✅  1180.SR      Full: -0.0295  |  Recent 30D: -0.4612
  ✅  2010.SR      Full: +0.0839  |  Recent 30D: +0.2656
  ✅  2222.SR      Full: +0.1251  |  Recent 30D: +0.2966
  ✅  AAPL         Full: +0.0211  |  Recent 30D: -0.4770
  ✅  AMZN         Full: +0.0092  |  Recent 30D: -0.4702
  ✅  BAC          Full: +0.0649  |  Recent 30D: -0.5622
  ✅  BZ=F         Full: +0.9362  |  Recent 30D: +0.8633
  ✅  CVX          Full: +0.4875  |  Recent 30D: +0.5300
  ✅  GC=F         Full: +0.0709  |  Recent 30D: -0.1783
  ✅  GOOGL        Full: +0.0017  |  Recent 30D: -0.4592
  ✅  GS           Full: -0.0048  |  Recent 30D: -0.5553
  ✅  JNJ          Full: -0.0969  |  Recent 30D: -0.3258
  ✅  JPM          Full: +0.0429  |  Recent 30D: -0.6067
  ✅  MCD          Full: -0.0839  |  Recent 30D: -0.0811
  ✅  MSFT         Full: +0.0055  |  Recent 30D: -0.0880
  ✅  NVDA         Full: +0.0338  |  Recent 30D: -0.3430
  ✅  PFE          Full: -0.0563  |  Recent 30D: -0.2057
  ✅  SLB     

In [8]:

itables.show(corr_df)

Loading ITables v2.5.2 from the internet... (need help?)


---
## Section 6 — Sector-Level Daily Aggregation

Instead of analyzing 30 individual stocks, this aggregates them by sector per day.  
This gives us a cleaner view of **sector-level behavior** over time.

Excluded from aggregation: `Oil & Energy` and `Benchmarks`  
(these are reference data, not sectors we're analyzing the impact on)

Output: one row per `(Date, Sector)` combination with averaged metrics.


In [9]:
def build_sector_summary(prices_df):

    exclude = {"Benchmarks", "Oil & Energy"}
    df = prices_df[~prices_df["Sector"].isin(exclude)].copy()

    summary = df.groupby(["Date", "Sector"]).agg(
        Avg_Daily_Return      = ("Daily_Return_%",      "mean"),
        Avg_Volatility_30D    = ("Volatility_30D",      "mean"),
        Avg_Cumulative_Return = ("Cumulative_Return_%",  "mean"),
        Avg_Volume_Spike      = ("Volume_Spike",         "mean"),
    ).reset_index().round(4)

    print(f"  ✅  Sector summary built: {len(summary):,} rows")
    return summary

# ── Run ───────────────────────────────────────────────────────────
sector_df = build_sector_summary(prices_df)

print(f"\n Sector summary shape : {sector_df.shape}")
itables.show(sector_df)


  ✅  Sector summary built: 3,758 rows

 Sector summary shape : (3758, 6)


Loading ITables v2.5.2 from the internet... (need help?)


---
## Section 7 — Data Validation

Before exporting, we run a quality check on every ticker.

**What we check:**
- `Row_Count` — how many trading days were returned (expected ~750 for 3y)
- `Date_From / Date_To` — confirms the date range looks correct
- `Null_Returns` — how many days have missing return values
- `Data_Quality` — auto-flag: `⚠️ Low` if rows < 400 or nulls > 20



In [10]:
def validate_and_report(prices_df, failed_tickers):

    report = prices_df.groupby("Ticker").agg(
        Name          = ("Name",           "first"),
        Sector        = ("Sector",         "first"),
        Row_Count     = ("Close",          "count"),
        Date_From     = ("Date",           "min"),
        Date_To       = ("Date",           "max"),
        Null_Returns  = ("Daily_Return_%", lambda x: x.isna().sum()),
        Null_Volume   = ("Volume",         lambda x: x.isna().sum()),
        Min_Close     = ("Close",          "min"),
        Max_Close     = ("Close",          "max"),
        Avg_Daily_Vol = ("Volume",         "mean"),
    ).reset_index()

    report["Date_From"] = report["Date_From"].astype(str)
    report["Date_To"]   = report["Date_To"].astype(str)
    report["Data_Quality"] = report.apply(
        lambda r: "⚠️ Low" if r["Row_Count"] < 400 or r["Null_Returns"] > 20
        else "✅ Good", axis=1
    )
    report = report.round(2)

    print(f"  {'Ticker':<12} {'Rows':>5}  {'From':<12} {'To':<12} {'Nulls':>5}  Quality")
    print(f"  {'─'*12} {'─'*5}  {'─'*12} {'─'*12} {'─'*5}  {'─'*10}")
    for _, r in report.iterrows():
        print(f"  {r['Ticker']:<12} {r['Row_Count']:>5}  {r['Date_From']:<12} {r['Date_To']:<12} {int(r['Null_Returns']):>5}  {r['Data_Quality']}")

    if failed_tickers:
        print(f"\n  ❌ Failed tickers (no data at all): {', '.join(failed_tickers)}")

    return report

# ── Run ───────────────────────────────────────────────────────────
validation_df = validate_and_report(prices_df, failed_list)


  Ticker        Rows  From         To           Nulls  Quality
  ──────────── ─────  ──────────── ──────────── ─────  ──────────
  1180.SR        745  2023-04-09   2026-04-09       1  ✅ Good
  2010.SR        745  2023-04-09   2026-04-09       1  ✅ Good
  2222.SR        746  2023-04-09   2026-04-09       1  ✅ Good
  AAPL           753  2023-04-10   2026-04-09       1  ✅ Good
  AMZN           753  2023-04-10   2026-04-09       1  ✅ Good
  BAC            753  2023-04-10   2026-04-09       1  ✅ Good
  BZ=F           757  2023-04-10   2026-04-10       1  ✅ Good
  CL=F           756  2023-04-10   2026-04-10       1  ✅ Good
  CVX            753  2023-04-10   2026-04-09       1  ✅ Good
  GC=F           756  2023-04-10   2026-04-10       1  ✅ Good
  GOOGL          753  2023-04-10   2026-04-09       1  ✅ Good
  GS             753  2023-04-10   2026-04-09       1  ✅ Good
  JNJ            753  2023-04-10   2026-04-09       1  ✅ Good
  JPM            753  2023-04-10   2026-04-09       1  ✅ Good
  M

In [11]:
# Full validation table
itables.show(validation_df)


Loading ITables v2.5.2 from the internet... (need help?)


---
## Section 8 — Export to CSV

All 5 files are saved to the `./data/` folder with **fixed filenames**.  
Re-running this notebook overwrites the existing files with fresh data.

**Next step:** Import these CSVs into SQL Server for ETL and EDA.


In [12]:
def save(df, filename, label):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"  💾  {label:<35} → {filename}  ({len(df):,} rows)")
    return path


save(prices_df,    "stock_prices.csv",     "Price History (main fact table)")
save(info_df,      "company_info.csv",     "Company Metadata (dimension table)")
save(corr_df,      "oil_correlations.csv", "Oil Correlation Analysis")
save(sector_df,    "sector_summary.csv",   "Sector Daily Aggregation")
save(validation_df,"data_validation.csv",  "Data Quality Report")

total_rows = len(prices_df) + len(info_df) + len(corr_df) + len(sector_df)

print("\n===========================================")
print("  EXPORT COMPLETE")
print(f"  Total rows saved : {total_rows:,}")
print(f"  Output folder    : ./{OUTPUT_DIR}/")
print("===========================================")


  💾  Price History (main fact table)     → stock_prices.csv  (22,578 rows)
  💾  Company Metadata (dimension table)  → company_info.csv  (30 rows)
  💾  Oil Correlation Analysis            → oil_correlations.csv  (29 rows)
  💾  Sector Daily Aggregation            → sector_summary.csv  (3,758 rows)
  💾  Data Quality Report                 → data_validation.csv  (30 rows)

  EXPORT COMPLETE
  Total rows saved : 26,395
  Output folder    : ./data/
